# DIA-MSの理論とファイル形式を理解する【論文再現シリーズ #2b】

## はじめに

前回（[#2a データ取得](notebook_02a_data_acquisition.ipynb)）でダウンロードしたデータの中身を詳しく見ていきます。この記事では、DIA-MSの基本理論と、RAW・mzMLファイル形式の違いを具体例とともに解説します。

> **📝 INFO**
>
> **この記事で行う処理**
> DIA（Data-Independent Acquisition）質量分析の理論的背景を学び、Thermo独自のRAWファイルとオープンなmzMLファイルの構造を比較します。実際のファイル内容をhexダンプやXML構造で確認し、以降のsage解析でmzMLを使用する理由を理解します。

In [ ]:
# 必要なライブラリをインポート
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import xml.etree.ElementTree as ET
from IPython.display import display, HTML, Image
import warnings
warnings.filterwarnings('ignore')

# プロジェクト設定
project_root = Path("/home/shizuku/labcode/article/Proteomics_drug_marker")
data_dir = project_root / "data" / "raw"

# 日本語フォント設定
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style("whitegrid")

print("✅ ライブラリのインポート完了")
print(f"📂 プロジェクトルート: {project_root}")
print(f"📁 データディレクトリ: {data_dir}")

## 🔬 DIA-MSとは？

本論文のRAWファイルは **DIA（Data-Independent Acquisition）** モードで測定されたデータです。データを解析する前に、DIA-MSの基本を押さえておきましょう。

In [ ]:
# DDA vs DIA の比較表を作成
acquisition_comparison = {
    "項目": [
        "選択方法",
        "再現性",
        "データ複雑性", 
        "定量性",
        "主な用途"
    ],
    "DDA（Data-Dependent Acquisition）": [
        "MS1スキャンで強度の高いイオンを選択してMS2を取得",
        "測定ごとに選択されるイオンが異なる → 再現性が低い",
        "MS2スペクトルが1ペプチドに対応 → 解析が比較的シンプル",
        "Missing valueが多い",
        "探索的な同定、ライブラリ構築"
    ],
    "DIA（Data-Independent Acquisition）": [
        "あらかじめ決めたm/z範囲を網羅的にスキャン",
        "全イオンを取得 → 再現性が高い",
        "MS2スペクトルが複数ペプチド由来の混合 → 専用ソフトウェアが必要",
        "Missing valueが少なく定量精度が高い",
        "大規模コホートの定量比較"
    ]
}

comparison_df = pd.DataFrame(acquisition_comparison)
display(HTML(comparison_df.to_html(index=False, escape=False)))

print("\n🎯 本論文でDIAモードが採用された理由:")
print("• 16患者 × 2条件（腫瘍/正常組織）の比較")
print("• 高い再現性で安定した定量値が必要")
print("• Missing valueの少ないデータセットが必要")

In [ ]:
# DIA-MSの概念図を作成
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# DDA概念図
ax1.set_title('DDA (Data-Dependent Acquisition)', fontsize=14, fontweight='bold')
# MS1 スキャン
ax1.barh([0], [100], color='lightblue', alpha=0.7, label='MS1 Full Scan')
# 選択的なMS2スキャン
selected_peaks = [20, 40, 80]
for i, peak in enumerate(selected_peaks):
    ax1.barh([i+1], [15], left=peak, color='orange', alpha=0.8, label='MS2 (selected)' if i==0 else '')
ax1.set_xlim(0, 120)
ax1.set_ylim(-0.5, 3.5)
ax1.set_xlabel('m/z range')
ax1.set_ylabel('Scan Level')
ax1.set_yticks([0, 1, 2, 3])
ax1.set_yticklabels(['MS1', 'MS2-1', 'MS2-2', 'MS2-3'])
ax1.legend()
ax1.text(60, -0.3, '↑ 強度上位のピークのみ選択', ha='center', color='red', fontsize=10)

# DIA概念図  
ax2.set_title('DIA (Data-Independent Acquisition)', fontsize=14, fontweight='bold')
# MS1 スキャン
ax2.barh([0], [100], color='lightblue', alpha=0.7, label='MS1 Full Scan')
# 連続的なDIA窓
dia_windows = [(0, 25), (20, 45), (40, 65), (60, 85), (80, 105)]
for i, (start, end) in enumerate(dia_windows):
    ax2.barh([i+1], [end-start], left=start, color='green', alpha=0.8, 
             label='DIA windows' if i==0 else '')
ax2.set_xlim(0, 120)
ax2.set_ylim(-0.5, 5.5)
ax2.set_xlabel('m/z range')
ax2.set_ylabel('Scan Level')
ax2.set_yticks(range(6))
ax2.set_yticklabels(['MS1'] + [f'DIA-{i+1}' for i in range(5)])
ax2.legend()
ax2.text(60, -0.3, '↑ 全m/z範囲を網羅的にスキャン', ha='center', color='green', fontsize=10)

plt.tight_layout()
plt.show()

print("📊 DDAとDIAの測定戦略の違い:")
print("• DDA: 強度上位のピークのみ選択的に測定 → 再現性に課題")
print("• DIA: 全m/z範囲を連続的な窓で網羅的に測定 → 高い再現性")

### 🔬 Orbitrap Exploris 480

本データの測定に使用された **Orbitrap Exploris 480**（Thermo Fisher Scientific）は、高分解能・高感度のOrbitrap型質量分析計です。DIAモードとの組み合わせにより、1回の測定で数千タンパク質を安定して定量できます。

In [ ]:
# Orbitrap Exploris 480の仕様情報
instrument_specs = {
    "項目": [
        "装置名",
        "メーカー",
        "分析部",
        "質量分解能",
        "質量精度",
        "イオン源",
        "DIA対応",
        "主な特徴"
    ],
    "仕様": [
        "Orbitrap Exploris 480",
        "Thermo Fisher Scientific",
        "Quadrupole + Orbitrap",
        "最大480,000 (FWHM at m/z 200)",
        "<1 ppm (external calibration)",
        "HESI, nano-ESI, APCI, APPI",
        "Yes (advanced DIA capabilities)",
        "高速・高感度・高分解能、Advanced Peak Determination (APD)機能"
    ]
}

specs_df = pd.DataFrame(instrument_specs)
display(HTML(specs_df.to_html(index=False, escape=False)))

print("\n🎯 DIAモードでの利点:")
print("• 高分解能により複雑なDIAスペクトルを高精度で測定")
print("• 高速スキャンにより多くのDIA窓を短時間で取得")
print("• APD機能により低強度ピークも正確に検出")

### 🔧 解析パイプラインへの影響

DIAデータはスペクトルが複雑なため、従来のDDA用サーチエンジン（Mascot, MaxQuantのAndromeda等）ではなく、**DIA専用の解析ツール**が必要です。

In [ ]:
# DIA解析ツールの比較
dia_tools_comparison = {
    "ツール名": [
        "DIA-NN",
        "sage-proteomics",
        "OpenSWATH",
        "Spectronaut",
        "MaxQuant"
    ],
    "ライセンス": [
        "非商用のみ",
        "MIT（商用OK）",
        "Apache 2.0（商用OK）",
        "商用ソフト",
        "非商用のみ"
    ],
    "主な特徴": [
        "深層学習予測、高検出率",
        "高速・軽量、Rust製",
        "オープンソース、統計重視",
        "商用最高品質、高機能",
        "DDA/DIA両対応"
    ],
    "本シリーズでの採用": [
        "❌（商用制限）",
        "✅（メイン使用）",
        "✅（補助使用）",
        "❌（有料）",
        "❌（商用制限）"
    ]
}

tools_df = pd.DataFrame(dia_tools_comparison)
display(HTML(tools_df.to_html(index=False, escape=False)))

print("\n🎯 本シリーズでsage-proteomicsを選択した理由:")
print("• MITライセンス → 商用利用完全OK")
print("• Rust製 → 高速・低メモリ使用量")
print("• シンプル → 設定・実行が容易")
print("• mzML対応 → オープンフォーマットで連携良好")

## 📁 ファイル形式: RAW と mzML

ダウンロードしたファイルの中身を実際に覗いてみましょう。

In [ ]:
# データディレクトリの確認
if data_dir.exists():
    print(f"📂 データディレクトリ: {data_dir}")
    
    # ファイル一覧を取得
    raw_files = list(data_dir.glob("*.raw"))
    mzml_files = list(data_dir.glob("*.mzML"))
    fasta_files = list(data_dir.glob("*.fasta"))
    
    print(f"\n📄 ファイル統計:")
    print(f"  RAWファイル: {len(raw_files)} 個")
    print(f"  mzMLファイル: {len(mzml_files)} 個")
    print(f"  FASTAファイル: {len(fasta_files)} 個")
    
    # 最初のいくつかのファイルを表示
    if raw_files:
        print(f"\n🔬 RAWファイル例（最初の3個）:")
        for i, f in enumerate(raw_files[:3]):
            size_mb = f.stat().st_size / 1024 / 1024
            print(f"  {i+1}. {f.name} ({size_mb:.1f} MB)")
    
    if mzml_files:
        print(f"\n📄 mzMLファイル例（最初の3個）:")
        for i, f in enumerate(mzml_files[:3]):
            size_mb = f.stat().st_size / 1024 / 1024
            print(f"  {i+1}. {f.name} ({size_mb:.1f} MB)")
            
else:
    print(f"⚠️ データディレクトリが見つかりません: {data_dir}")
    print("まず #2a データ取得を完了してください。")

### 🔍 .raw ファイル（Thermo独自バイナリ）

`.raw` はThermo Fisher Scientific独自のバイナリ形式です。ファイルの先頭をhexダンプすると、Thermo独自の構造が見えます：

In [ ]:
# RAWファイルのヘッダ部分を確認（もしファイルが存在すれば）
def examine_raw_file_header(file_path, num_bytes=128):
    """RAWファイルのヘッダ部分をhexダンプ風に表示"""
    if not file_path.exists():
        print(f"❌ ファイルが見つかりません: {file_path}")
        return
    
    try:
        with open(file_path, 'rb') as f:
            header = f.read(num_bytes)
        
        print(f"🔍 RAWファイルヘッダ解析: {file_path.name}")
        print(f"ファイルサイズ: {file_path.stat().st_size / 1024 / 1024:.1f} MB")
        print("\n📄 ヘッダ部分（先頭128バイト）:")
        
        # hexダンプ風に表示
        for i in range(0, min(len(header), num_bytes), 16):
            chunk = header[i:i+16]
            hex_part = ' '.join(f'{b:02x}' for b in chunk)
            ascii_part = ''.join(chr(b) if 32 <= b <= 126 else '.' for b in chunk)
            print(f"{i:08x}: {hex_part:<48} {ascii_part}")
        
        # Thermoファイルの特徴的な文字列を検索
        header_str = header.decode('utf-8', errors='ignore')
        thermo_indicators = ['Finnigan', 'Xcalibur', 'Thermo']
        found_indicators = [ind for ind in thermo_indicators if ind in header_str]
        
        if found_indicators:
            print(f"\n✅ Thermoファイル識別子検出: {', '.join(found_indicators)}")
        else:
            print("\n🤔 Thermoファイル識別子が見つかりませんでした")
            
    except Exception as e:
        print(f"❌ ファイル読み込みエラー: {e}")

# 最初のRAWファイルを調べる
if raw_files:
    examine_raw_file_header(raw_files[0])
else:
    print("📝 RAWファイルのヘッダ構造例:")
    print("""
00000000: 01a1 4600 6900 6e00 6e00 6900 6700 6100  ..F.i.n.n.i.g.a.
00000010: 6e00 ...                                   n.
00000030: 5800 6300 6100 6c00 6900 6200 7500 7200  X.c.a.l.i.b.u.r.
00000040: 5f00 5300 7900 7300 7400 6500 6d00 ...   _.S.y.s.t.e.m.
00000060: 0000 5400 6800 6500 7200 6d00 6f00 ...   ..T.h.e.r.m.o.

特徴的な識別子:
• Finnigan — Thermo RAW形式のマジックバイト
• Xcalibur_System — 測定制御ソフトウェア名
• Thermo — メーカー名
""")

### 📄 .mzML ファイル（オープンXML形式）

`.mzML` はRAWファイルをProteoWizardで変換したオープンフォーマットです。XML形式なので、テキストエディタでも内容を確認できます。

In [ ]:
# mzMLファイルのメタデータを解析
def examine_mzml_metadata(file_path):
    """mzMLファイルのメタデータを抽出して表示"""
    if not file_path.exists():
        print(f"❌ ファイルが見つかりません: {file_path}")
        return
    
    try:
        print(f"📄 mzMLファイル解析: {file_path.name}")
        print(f"ファイルサイズ: {file_path.stat().st_size / 1024 / 1024:.1f} MB")
        
        # XMLの先頭部分を読み取り（大きなファイルのため部分読み取り）
        with open(file_path, 'r', encoding='utf-8') as f:
            # 最初の10,000文字を読み取り
            header_content = f.read(10000)
        
        print("\n🔍 XMLヘッダ構造の例:")
        lines = header_content.split('\n')[:30]  # 最初の30行
        for i, line in enumerate(lines):
            if i < 20:  # 最初の20行のみ表示
                print(f"{i+1:2d}: {line[:80]}{'...' if len(line) > 80 else ''}")
        
        # 重要な情報を抽出
        metadata = {}
        
        # 装置名を検索
        if "Orbitrap Exploris" in header_content:
            metadata["装置"] = "Orbitrap Exploris 480"
        
        # ソフトウェア情報を検索
        if "Xcalibur" in header_content:
            metadata["制御ソフト"] = "Xcalibur"
        if "pwiz" in header_content:
            metadata["変換ソフト"] = "ProteoWizard"
        
        # スキャン数を検索
        import re
        scan_match = re.search(r'count="([0-9]+)"', header_content)
        if scan_match:
            metadata["総スキャン数"] = scan_match.group(1)
        
        if metadata:
            print("\n📊 抽出されたメタデータ:")
            for key, value in metadata.items():
                print(f"  {key}: {value}")
        
    except Exception as e:
        print(f"❌ ファイル解析エラー: {e}")

# 最初のmzMLファイルを調べる
if mzml_files:
    examine_mzml_metadata(mzml_files[0])
else:
    print("📝 mzMLファイルのメタデータ構造例:")
    print("""
<?xml version="1.0" encoding="utf-8"?>
<mzML xmlns="http://psi.hupo.org/ms/mzml" version="1.1.0">
  <cvList count="2">
    <cv id="MS" fullName="Proteomics Standards Initiative Mass Spectrometry Ontology"/>
    <cv id="UO" fullName="Unit Ontology"/>
  </cvList>
  
  <fileDescription>
    <fileContent>
      <cvParam cvRef="MS" accession="MS:1000579" name="MS1 spectrum"/>
      <cvParam cvRef="MS" accession="MS:1000580" name="MSn spectrum"/>
    </fileContent>
  </fileDescription>
  
  <referenceableParamGroup id="CommonInstrumentParams">
    <cvParam name="Orbitrap Exploris 480" />
    <cvParam name="instrument serial number" value="MA10127C" />
  </referenceableParamGroup>
  
  <software id="Xcalibur" version="3.1-3.1.231.6/3.1.279.9" />
  <software id="pwiz" version="3.0.21257" />
  
  <run id="CRC04-T" startTimeStamp="2022-12-22T10:33:42Z">
    <spectrumList count="51701">
""")

In [ ]:
# mzMLとRAWファイルの比較表
file_format_comparison = {
    "項目": [
        "形式",
        "可読性", 
        "ファイルサイズ（例）",
        "DIA解析ツールへの入力",
        "汎用性",
        "メタデータアクセス",
        "圧縮効率"
    ],
    ".raw": [
        "Thermo独自バイナリ",
        "ツールがないと読めない",
        "約1.3 GB",
        "ツールによっては直接読み込み可能", 
        "Thermo製ツール限定",
        "専用ライブラリが必要",
        "独自圧縮"
    ],
    
    ".mzML": [
        "オープンXML",
        "テキストエディタで確認可能",
        "約0.9 GB",
        "直接読み込み可能",
        "あらゆる解析ツールで利用可能",
        "XMLパーサーで容易にアクセス",
        "zlib + Base64"
    ]
}

format_df = pd.DataFrame(file_format_comparison)
display(HTML(format_df.to_html(index=False, escape=False)))

print("\n🎯 本シリーズでmzMLを使用する理由:")
print("✅ sage・OpenMSなど主要な解析ツールで読み込み可能")
print("✅ XMLパーサーでメタデータに容易にアクセス")
print("✅ オープンフォーマットで将来性が高い")
print("✅ 圧縮効率が良く、ファイルサイズが小さい")

print("\n⚠️ 注意点:")
print("• ProteomeXchangeではRAWファイルのみ公開されているケースが大半")
print("• その場合はRAW → mzML変換が必要（次章で詳解）")

### 🔬 スペクトルデータの構造

mzMLファイル内の実際のスペクトルデータがどのように格納されているかを確認してみましょう。

In [ ]:
# スペクトルデータの構造例を表示
print("📊 mzMLファイル内のスペクトルデータ構造例:")
print("""
<spectrum index="0" id="controllerType=0 controllerNumber=1 scan=1">
  <cvParam name="MS1 spectrum" />
  <cvParam name="ms level" value="1" />
  <cvParam name="positive scan" />
  <cvParam name="centroid spectrum" />
  <cvParam name="base peak m/z" value="548.2862601" />
  <cvParam name="total ion current" value="1.96729888e08" />
  <cvParam name="filter string" 
           value="FTMS + c NSI Full ms [495.0000-865.0000]" />
  
  <binaryDataArrayList count="2">
    <binaryDataArray encodedLength="...">
      <cvParam name="m/z array" />
      <cvParam name="64-bit float" />
      <cvParam name="zlib compression" />
      <binary>eJzt2M1... (Base64エンコード)</binary>
    </binaryDataArray>
    
    <binaryDataArray encodedLength="...">
      <cvParam name="intensity array" />
      <cvParam name="64-bit float" />
      <cvParam name="zlib compression" />
      <binary>eJzt2M1... (Base64エンコード)</binary>
    </binaryDataArray>
  </binaryDataArrayList>
</spectrum>
""")

# データエンコーディングの流れを図解
encoding_steps = [
    "1. 生データ（m/z配列, intensity配列）",
    "↓",
    "2. 64-bit float配列に変換", 
    "↓",
    "3. zlib圧縮",
    "↓",
    "4. Base64エンコード",
    "↓",
    "5. XMLテキストとして保存"
]

print("\n🔄 スペクトルデータのエンコーディング手順:")
for step in encoding_steps:
    print(f"  {step}")

print("\n⚙️ sage等の解析ツールは以下の手順でデータを読み込みます:")
print("  1. XMLをパース")
print("  2. Base64デコード")
print("  3. zlib展開")
print("  4. 64-bit float配列として解釈")
print("  5. m/z配列とintensity配列として使用")

## 📊 ファイルサイズとデータ効率の比較

In [ ]:
# ファイルサイズの比較（もしファイルが存在すれば）
if raw_files and mzml_files:
    print("📊 ファイルサイズ比較（先頭5ファイル）:")
    print("-" * 60)
    print(f"{'ファイル名':<20} {'RAW (MB)':<12} {'mzML (MB)':<12} {'比率':<8}")
    print("-" * 60)
    
    size_data = []
    for i in range(min(5, len(raw_files), len(mzml_files))):
        raw_size = raw_files[i].stat().st_size / 1024 / 1024
        mzml_size = mzml_files[i].stat().st_size / 1024 / 1024
        ratio = mzml_size / raw_size
        
        base_name = raw_files[i].stem
        print(f"{base_name:<20} {raw_size:<12.1f} {mzml_size:<12.1f} {ratio:<8.2f}")
        
        size_data.append({
            "ファイル": base_name,
            "RAW_MB": raw_size,
            "mzML_MB": mzml_size,
            "比率": ratio
        })
    
    if size_data:
        # 平均値を計算
        avg_raw = np.mean([d['RAW_MB'] for d in size_data])
        avg_mzml = np.mean([d['mzML_MB'] for d in size_data])
        avg_ratio = avg_mzml / avg_raw
        
        print("-" * 60)
        print(f"{'平均':<20} {avg_raw:<12.1f} {avg_mzml:<12.1f} {avg_ratio:<8.2f}")
        
        # 可視化
        fig, ax = plt.subplots(1, 1, figsize=(10, 6))
        
        files = [d['ファイル'] for d in size_data]
        raw_sizes = [d['RAW_MB'] for d in size_data]
        mzml_sizes = [d['mzML_MB'] for d in size_data]
        
        x = np.arange(len(files))
        width = 0.35
        
        ax.bar(x - width/2, raw_sizes, width, label='RAW', alpha=0.8)
        ax.bar(x + width/2, mzml_sizes, width, label='mzML', alpha=0.8)
        
        ax.set_xlabel('ファイル')
        ax.set_ylabel('ファイルサイズ (MB)')
        ax.set_title('RAW vs mzML ファイルサイズ比較')
        ax.set_xticks(x)
        ax.set_xticklabels(files, rotation=45)
        ax.legend()
        
        plt.tight_layout()
        plt.show()
        
        print(f"\n📈 サイズ比較結果:")
        print(f"• mzMLは平均してRAWファイルの{avg_ratio:.1f}倍のサイズ")
        if avg_ratio < 1:
            print(f"• mzMLの方が{(1-avg_ratio)*100:.1f}%小さい（圧縮効率が良い）")
        else:
            print(f"• mzMLの方が{(avg_ratio-1)*100:.1f}%大きい")

else:
    print("📝 一般的なファイルサイズ比較例:")
    print("• RAWファイル: 1.2-1.5 GB")
    print("• mzMLファイル: 0.8-1.2 GB")
    print("• mzMLはzlib圧縮により通常20-30%小さくなる")

## 🎯 まとめ

DIAモードの理論的背景と、RAW・mzMLファイル形式の違いを具体例で確認しました。

In [ ]:
# まとめのポイント
summary_points = {
    "DIA-MSの利点": [
        "高い再現性 → 大規模コホート研究に最適",
        "網羅的測定 → Missing valueが少ない",
        "定量精度 → 正確な比較解析が可能",
        "Orbitrap Exploris 480 → 高分解能・高感度測定"
    ],
    "ファイル形式の選択": [
        "RAW → Thermo独自、ツール限定",
        "mzML → オープンXML、汎用性高い",
        "sage-proteomics → mzML直接対応",
        "圧縮効率 → mzMLの方が通常小さい"
    ],
    "解析ツールの選択": [
        "sage-proteomics → MIT、商用OK、高速",
        "DIA専用 → 複雑スペクトル対応",
        "Rust製 → 低メモリ、高性能",
        "mzML対応 → オープンフォーマット活用"
    ]
}

print("📋 重要ポイントまとめ")
print("=" * 50)

for category, points in summary_points.items():
    print(f"\n🎯 {category}:")
    for point in points:
        print(f"  • {point}")

print("\n🚀 次のステップ:")
print("• mzMLファイルがない場合 → [#3a RAW→mzML変換](notebook_03a_convert_basics.ipynb)")
print("• mzMLファイルがある場合 → [#4a Sage基礎](notebook_04a_sage_fundamentals.ipynb)")
print("• DIAの高い再現性とmzMLの汎用性を活かした解析を開始")

print("\n#バイオインフォマティクス #プロテオミクス #DIA-MS #ファイル形式 #labcode")